# Dataset + Dataloader Guide

This notebook is only for the dataset and dataloader path.

It shows:
- how `ArtRestorationDataset` reads train and validation images
- how train corruption differs from validation corruption
- what one sample and one batch look like
- how the sampler changes order by epoch and supports resume


In [ ]:
from pathlib import Path

# Values to set up.
# These defaults match the current training config.
TRAIN_DIR = Path("/nfs/roberts/project/cpsc4520/cpsc4520_ckk25/data/train")
VAL_DIR = Path("/nfs/roberts/project/cpsc4520/cpsc4520_ckk25/data/val")
RESOLUTION = 512
BATCH_SIZE = 4
NUM_WORKERS = 0  # keep 0 in the notebook so behavior is easier to inspect
SAMPLER_SEED = 42
CORRUPTION_SEED = 42
NUM_SHOW = 4


In [ ]:
from IPython.display import display
import torch
from omegaconf import OmegaConf
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

from src.config import CorruptionConfig
from src.dataset import ArtRestorationDataset, StatefulEpochSampler, build_wikiart_dataloader


def load_corruption_config() -> CorruptionConfig:
    cfg_path = Path("src/corruption/configs/default.yaml")
    cfg_dict = OmegaConf.to_container(OmegaConf.load(cfg_path), resolve=True)
    return CorruptionConfig(**cfg_dict)


def damage_map(mask: torch.Tensor) -> torch.Tensor:
    return mask.max(dim=0).values.unsqueeze(0).repeat(3, 1, 1)


def show_sample(sample: dict, title: str) -> None:
    tiles = [sample["clean"], sample["corrupted"], damage_map(sample["mask"])]
    grid = make_grid(tiles, nrow=3, padding=8)
    print(title)
    print({
        "path": sample["path"],
        "genre": sample["genre"],
        "clean_shape": tuple(sample["clean"].shape),
        "corrupted_shape": tuple(sample["corrupted"].shape),
        "mask_shape": tuple(sample["mask"].shape),
        "corruption_seed": sample["corruption_seed"],
    })
    display(to_pil_image(grid))


def show_batch(batch: dict, title: str, num_show: int = 4) -> None:
    count = min(num_show, batch["clean"].shape[0])
    tiles = []
    for i in range(count):
        tiles.extend([
            batch["clean"][i],
            batch["corrupted"][i],
            damage_map(batch["mask"][i]),
        ])
    grid = make_grid(tiles, nrow=3, padding=8)
    print(title)
    print({
        "clean": tuple(batch["clean"].shape),
        "corrupted": tuple(batch["corrupted"].shape),
        "mask": tuple(batch["mask"].shape),
        "paths_shown": batch["path"][:count],
    })
    display(to_pil_image(grid))


assert TRAIN_DIR.exists(), f"Missing train dir: {TRAIN_DIR}"
assert VAL_DIR.exists(), f"Missing val dir: {VAL_DIR}"

corruption_cfg = load_corruption_config()


In [ ]:
train_dataset = ArtRestorationDataset(
    image_dir=str(TRAIN_DIR),
    resolution=RESOLUTION,
    corruption_config=corruption_cfg,
    split="train",
    corruption_seed=CORRUPTION_SEED,
    deterministic_corruption=False,
)

val_dataset = ArtRestorationDataset(
    image_dir=str(VAL_DIR),
    resolution=RESOLUTION,
    corruption_config=corruption_cfg,
    split="eval",
    corruption_seed=CORRUPTION_SEED,
    deterministic_corruption=True,
)

print("train summary:", train_dataset.split_summary())
print("val summary:", val_dataset.split_summary())


In [ ]:
train_a = train_dataset[0]
train_b = train_dataset[0]
val_a = val_dataset[0]
val_b = val_dataset[0]

print("train sample changes corruption across fetches:", not torch.equal(train_a["corrupted"], train_b["corrupted"]))
print("val sample stays deterministic across fetches:", torch.equal(val_a["corrupted"], val_b["corrupted"]))

show_sample(train_a, "Train sample: clean | corrupted | max damage map")
show_sample(val_a, "Validation sample: clean | corrupted | max damage map")


In [ ]:
train_dataset_dl, train_loader, train_sampler = build_wikiart_dataloader(
    train_dir=str(TRAIN_DIR),
    val_dir=str(VAL_DIR),
    resolution=RESOLUTION,
    corruption_config=corruption_cfg,
    split="train",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    sampler_seed=SAMPLER_SEED,
    corruption_seed=CORRUPTION_SEED,
    deterministic_corruption=False,
    shuffle=True,
    drop_last=True,
    pin_memory=False,
    persistent_workers=False,
)

val_dataset_dl, val_loader, val_sampler = build_wikiart_dataloader(
    train_dir=str(TRAIN_DIR),
    val_dir=str(VAL_DIR),
    resolution=RESOLUTION,
    corruption_config=corruption_cfg,
    split="eval",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    sampler_seed=SAMPLER_SEED,
    corruption_seed=CORRUPTION_SEED,
    deterministic_corruption=True,
    shuffle=False,
    drop_last=False,
    pin_memory=False,
    persistent_workers=False,
)

train_batch = next(iter(train_loader))
val_batch = next(iter(val_loader))

show_batch(train_batch, "Train batch: rows are samples, columns are clean | corrupted | damage map", num_show=NUM_SHOW)
show_batch(val_batch, "Validation batch: rows are samples, columns are clean | corrupted | damage map", num_show=NUM_SHOW)


In [ ]:
print("first 10 indices in epoch 0:", train_sampler.current_order()[:10])
train_sampler.set_epoch(1)
print("first 10 indices in epoch 1:", train_sampler.current_order()[:10])

resume_demo = StatefulEpochSampler(train_dataset_dl, shuffle=True, seed=train_sampler.seed)
resume_iter = iter(resume_demo)
seen = [next(resume_iter) for _ in range(5)]
state = resume_demo.state_dict()

resumed = StatefulEpochSampler(train_dataset_dl, shuffle=True, seed=train_sampler.seed)
resumed.load_state_dict(state)

print("seen before checkpoint:", seen)
print("remaining after resume starts with:", resumed.remaining_indices()[:10])
print("resume check passed:", resumed.seen_indices() == seen)


## What To Look For

- Train data is read from `TRAIN_DIR`; validation data is read from `VAL_DIR`.
- The train dataset should show different corruptions when you fetch the same index twice.
- The validation dataset should show the same corruption when you fetch the same index twice.
- Batch tensors should have shapes `(B, 3, H, W)` for images and `(B, K, H, W)` for masks.
- The sampler order should change when you call `set_epoch(1)`.
- The sampler resume demo should show that the saved state continues from the correct position.
